## Installs

In [ ]:
!pip install -U torch torchvision
!pip install transformers datasets tqdm pandas scipy

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip install --force-reinstall --no-cache-dir typing_extensions==4.11.0
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
## Sometimes needed in runpod to make sure it goes to the network volumne
import os

os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"
os.environ["HF_HOME"] = "/workspace/hf_home"

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import CLIPForImageClassification
from datasets import load_from_disk, concatenate_datasets
import pandas as pd
import numpy as np
import os
from collections import defaultdict
import copy
from typing import Optional
import gc

## Configuration

In [ ]:
# Custom Datasets Configuration
num_class = [47, 10, 43, 10, 45, 196, 397, 10]
vals = [47, 10, 43, 10, 45, 196, 397, 10]
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]
size_nums = [float('inf')]

# Regular for CLIP
domain = "Base_Fine_Tuned" # "Base_Fine_Tuned" | "Fine_Tuned_Layer_Skipping"
transformation = "Standard" # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
model_name = "CLIP_ViT_Vision"
folder = f"./Results/Single_Class_W/{domain}/Entire_Transformation_Matrix_W"
os.makedirs(folder, exist_ok=True)
refer = CLIPForImageClassification.from_pretrained("openai/clip-vit-base-patch32")
indices = [i for i in range(refer.vision_model.encoder.config.num_hidden_layers)]
device = "cuda" if torch.cuda.is_available() else "cpu"

## Classes Prep

In [ ]:
def collate_fn(batch):
    images = torch.stack([example["pixel_values"] for example in batch])
    labels = torch.tensor([example["label"] for example in batch])
    
    return {
        "pixel_values": images,
        "labels": labels
    }

In [ ]:
class Hooks(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.CLS = []
    
    def forward(self, images):
        self.CLS = []
        hidden_states = self.model.vision_model.embeddings(images)
        hidden_states = self.model.vision_model.pre_layrnorm(hidden_states)

        batch_size, seq_len, _ = hidden_states.shape
        attention_mask = torch.ones((batch_size, seq_len), dtype=torch.bool, device=hidden_states.device)
        attention_mask = attention_mask[:, None, None, :]
        casual_attention_mask = None

        for encoder_layer in self.model.vision_model.encoder.layers:
            layer_outputs = encoder_layer(hidden_states, attention_mask, casual_attention_mask, output_attentions=False)
            hidden_states = layer_outputs[0]
            self.CLS.append(hidden_states[:, 0, :])
        
        return self.CLS

In [ ]:
# https://github.com/huggingface/transformers/blob/main/src/transformers/models/clip/modeling_clip.py
class Augmented(torch.nn.Module):
    def __init__(self, model, classifier=None, W=None, transform_stage=-1):
        super().__init__()
        self.model = model
        self.classifier = classifier if classifier is not None else torch.nn.Linear(self.model.config.vision_config.hidden_size, num_classes)
        self.W = torch.from_numpy(W.astype(np.float32)).to(device) if W is not None else None
        self.transform_stage = transform_stage
    
    def forward(self, images):
        hidden_states = self.model.vision_model.embeddings(images)
        hidden_states = self.model.vision_model.pre_layrnorm(hidden_states)

        batch_size, seq_len, _ = hidden_states.shape
        attention_mask = torch.ones((batch_size, seq_len), dtype=torch.bool, device=hidden_states.device)
        attention_mask = attention_mask[:, None, None, :]
        casual_attention_mask = None

        for i, encoder_layer in enumerate(self.model.vision_model.encoder.layers):
            layer_outputs = encoder_layer(hidden_states, attention_mask, casual_attention_mask, output_attentions=False)
            hidden_states = layer_outputs[0]
            if i == self.transform_stage:
                if self.W is None:
                    self.W = torch.eye(hidden_states.shape[-1], device=hidden_states.device, dtype=hidden_states.dtype)
                cls = hidden_states[:, 0, :]
                cls = cls @ self.W
                hidden_states[:, 0, :] = cls
                break
        
        hidden_states = self.model.vision_model.post_layernorm(hidden_states[:, 0, :])
        logits = self.classifier(hidden_states)

        return logits, hidden_states # logits, post-layernorm cls

In [ ]:
def cosineSimilarity(fine_tuned_cls, aug_cls):
    eps = 1e-8
    out_aug = F.normalize(aug_cls, dim=1, eps=eps)
    out_fine = F.normalize(fine_tuned_cls, dim=1, eps=eps)

    cos_sim = (out_aug * out_fine).sum(dim=1).mean().item()
    return cos_sim

## Evaluation

In [ ]:
def extract_vectors(train_loader, base_H, fine_tuned_H):
    Z0 = {i: [] for i in indices}
    Z1 = []

    with torch.no_grad():
        for batch in tqdm(train_loader, desc="Extracting"):
            images = batch["pixel_values"].to(device, non_blocking=True)

            if domain == "Fine_Tuned_Layer_Skipping":
                out_fine_tuned = fine_tuned_H(images)
                
                for i in indices:
                    Z0[i].append(out_fine_tuned[i].float().cpu())
                Z1.append(out_fine_tuned[-1].float().cpu())
            else:
                out_base = base_H(images)
                out_fine_tuned = fine_tuned_H(images)

                for i in indices:
                    Z0[i].append(out_base[i].float().cpu())
                Z1.append(out_fine_tuned[-1].float().cpu())

    Z1_last = torch.cat(Z1)
    Z1 = Z1_last.cpu().numpy()

    W = {}
    resid = {}

    for i, val in Z0.items():
        val = torch.cat(val)
        val = val.cpu().numpy()
        W[i], resid[i], _, _ = np.linalg.lstsq(val, Z1, rcond=None)
    
    return W, resid

def augment_models(transform_type, f_t, W=None, linear_probe=None):
    if W is None:
        W = {i: None for i in indices}
    aug = {}
    for i in indices:
        reference = copy.deepcopy(refer)
        if domain == "Fine_Tuned_Layer_Skipping":
            reference = copy.deepcopy(f_t.model)
        if transform_type == "Standard":
            model = Augmented(copy.deepcopy(reference), classifier=f_t.classifier, W=W[i], transform_stage=i)
        elif transform_type == "Base_Fine_Tuned_Classifier":
            model = Augmented(copy.deepcopy(reference), classifier=f_t.classifier, transform_stage=i)
        elif transform_type == "Base_Linear_Probe":
            model = Augmented(copy.deepcopy(reference), classifier=linear_probe.classifier, transform_stage=i)
        model = model.eval().to(device)
        aug[i] = model
    return aug

def evaluate(aug, test_loader, base, fine_tuned):
    correct_base = 0
    correct_fine_tuned = 0
    total_samples = 0

    correct = {i: 0 for i in indices}
    co_sim_cls = {i: [] for i in indices}

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            images = batch["pixel_values"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            total_samples += labels.size(0)
    
            # Base Model
            logits_base, cls_base = base(images)
            predicted = logits_base.argmax(dim=1)
            correct_base += (predicted == labels).sum().item()
    
            # Fine-Tuned Model
            logits_fine_tuned, cls_fine_tuned = fine_tuned(images)
            predicted = logits_fine_tuned.argmax(dim=1)
            correct_fine_tuned += (predicted == labels).sum().item()
    
            # Augmented Models
            for i in indices:
                logits_aug, cls_aug = aug[i](images)
                predicted = logits_aug.argmax(dim=1)
                correct[i] += (predicted == labels).sum().item()
                co_sim_cls[i].append(cosineSimilarity(cls_fine_tuned, cls_aug))

    base_acc = correct_base / total_samples
    fine_tuned_acc = correct_fine_tuned / total_samples

    for i in indices:
        correct[i] = correct[i] / total_samples
        co_sim_cls[i] = np.mean(co_sim_cls[i])

    print(f"Augmented {model_name} on {dataset_name} Results")
    for i in indices:
        print(f"\tAugmented {i} - Last ({indices[-1]}) Layer Accuracy: {correct[i]}")
        print(f"\tAverage Cosine Similarity of CLS Token of Augmented {i} Layer: {co_sim_cls[i]:.4f}")
    print(f"Base Accuracy: {base_acc:.4f}")
    print(f"Fine-Tuned Accuracy: {fine_tuned_acc:.4f}")

    return base_acc, fine_tuned_acc, correct, co_sim_cls

def save_results(correct, co_sim_cls, transformation, long_train_num, folder, train_size=None, save_W=False, W=None, resid=None):
    name = f"{transformation}_"
    folder = folder
    os.makedirs(folder, exist_ok=True)

    data = {
        'Classification_Accuracy': [correct[i] for i in indices],
        'CLS_Cosine_Similarity': [co_sim_cls[i] for i in indices]
    }

    if transformation == "Standard":
        data["Train_Data_Size"] = [train_size] * len(indices),
        data["Residuals"] = [resid[i] for i in indices],
        name += f"{train_size}_"
    if save_W:
        data["W"] = [W[i] for i in indices]

    df = pd.DataFrame(data, index=indices)
    name += f"Results_{long_train_num}.json"
    path = os.path.join(folder, name)
    df.to_json(path, orient="records", indent=2)


for k in range(0, len(dataset_name)):
    # Custom Datasets Configuration
    num_class = [47, 10, 43, 10, 45, 196, 397, 10]
    vals = [47, 10, 43, 10, 45, 196, 397, 10]
    dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]
    size_nums = [float('inf')]

    # Regular for CLIP
    domain = "Base_Fine_Tuned" # "Base_Fine_Tuned" | "Fine_Tuned_Layer_Skipping"
    transformation = "Standard" # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
    model_name = "CLIP_ViT_Vision"
    folder = f"./Results/Single_Class_W/{domain}/Entire_Transformation_Matrix_W"
    os.makedirs(folder, exist_ok=True)
    refer = CLIPForImageClassification.from_pretrained("openai/clip-vit-base-patch32")
    indices = [i for i in range(refer.vision_model.encoder.config.num_hidden_layers)]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Datasets
    train = load_from_disk(f'/workspace/preprocessed/{dataset_name[k]}/train_processed')
    val = load_from_disk(f'/workspace/preprocessed/{dataset_name[k]}/val_processed')
    test = load_from_disk(f'/workspace/preprocessed/{dataset_name[k]}/test_processed')

    train.set_format(type='torch', columns=["image", "label", "pixel_values"])
    val.set_format(type='torch', columns=["image", "label", "pixel_values"])
    test.set_format(type='torch', columns=["image", "label", "pixel_values"])

    test_loader = DataLoader(test, batch_size=32, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

    # Initialize Models
    num_classes = num_class[k]

    base = Augmented(copy.deepcopy(refer))
    base = base.eval().to(device)
    base_H = Hooks(copy.deepcopy(refer)).to(device)

    folder_path = os.path.join(folder, dataset_name[k])
    for i in range(1,6):
        # Fine-Tuned Prep
        f_t = Augmented(copy.deepcopy(refer))
        f_t.load_state_dict(torch.load(f"./Long_Training/Fine_Tuned/{dataset_name[k]}/best_{model_name}_{dataset_name[k]}_Long_Train_{i}.pt"))
        fine_tuned = Augmented(copy.deepcopy(f_t.model), f_t.classifier)
        fine_tuned = fine_tuned.eval().to(device)

        fine_tuned_H = Hooks(f_t.model).to(device)

        # Train Prep
        train_size= len(train)
        train_loader = DataLoader(train, batch_size=64, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=8, persistent_workers=True)

        print(f"Results for {model_name} on {dataset_name[k]}: {train_size} Training Images")

        # Testing Portion
        W, resid = extract_vectors(train_loader, base_H, fine_tuned_H)
        aug = augment_models(transformation, f_t, W=W)
        base_acc, fine_tuned_acc, correct, co_sim_cls = evaluate(aug, test_loader, base, fine_tuned)
        save_results(correct, co_sim_cls, transformation, i, folder_path, train_size=train_size, save_W=True, W=W, resid=resid)